In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import json
from tqdm import tqdm
import viser
import time

from lac.perception.segmentation import SemanticClasses, UnetSegmentation
from lac.slam.semantic_feature_tracker import SemanticFeatureTracker
from lac.slam.frontend import Frontend
from lac.slam.backend import Backend
from lac.util import load_data, load_images, load_stereo_images
import lac.params as params

%load_ext autoreload
%autoreload 2

# Load some data


In [ ]:
# data_path = "/home/shared/data_raw/LAC/segmentation/semantics_map1_preset2_recovery_agent"
data_path = "/home/shared/data_raw/LAC/runs/2025-05-28_11-59-12"
initial_pose, lander_pose, poses, imu_data, cam_config, json_data = load_data(data_path)
config = json.load(open("../configs/nine_loops.json"))
print(f"Loaded {len(poses)} poses")

In [ ]:
# images = load_images(data_path, cameras=["FrontLeft", "FrontRight"], start_frame=0, end_frame=10000)
left_imgs, right_imgs = load_stereo_images(data_path, start_frame=0, end_frame=2000)
images = {"FrontLeft": left_imgs, "FrontRight": right_imgs}

# Initialize


In [ ]:
START_FRAME = 80
# START_FRAME = 1650

feature_tracker = SemanticFeatureTracker(cam_config)
frontend = Frontend(feature_tracker, initial_pose=initial_pose)
backend = Backend(poses[START_FRAME], feature_tracker, config["loop_closure"])

init_data = {
    "step": START_FRAME,
    "FrontLeft": left_imgs[START_FRAME],
    "FrontRight": right_imgs[START_FRAME],
    "imu": imu_data[START_FRAME],
}

frontend.initialize(init_data)

# Run


In [ ]:
END_FRAME = START_FRAME + 2

eval_poses = [poses[START_FRAME]]
current_pose = poses[START_FRAME]

for frame in tqdm(range(START_FRAME + 2, END_FRAME + 2, 2)):
    data = {
        "step": frame,
        "FrontLeft": left_imgs[frame],
        "FrontRight": right_imgs[frame],
        "imu": imu_data[frame],  # TODO: change to imu_measurements
        "prev_pose": current_pose,
    }
    data = frontend.process_frame(data)
    backend.update(data)
    eval_poses.append(poses[frame])
    current_pose = backend.get_trajectory()[-1]

## Depth point cloud


In [ ]:
points = data["tracked_points"].points_local
pixels = data["tracked_points"].points
colors = np.repeat(
    left_imgs[END_FRAME][pixels[:, 1].astype(int), pixels[:, 0].astype(int)][:, np.newaxis],
    3,
    axis=1,
)

In [ ]:
display(Image.fromarray(left_imgs[END_FRAME], mode="L"))

In [ ]:
colors = {
    1: [1.0, 0.0, 0.0],  # red for rocks
    2: [1.0, 0.843, 0.0],  # gold for lander
    3: [0.5, 0.5, 0.5],  # gray for ground
    4: [0.0, 0.0, 0.0],  # black for sky
}

In [ ]:
depth_colors = plt.cm.plasma(np.clip(1 / data["tracked_points"].depths, 0, 1))[:, :3] * 255
# semantic_colors = plt.cm.hsv(data["tracked_points"].labels / 4)[:, :3] * 255
semantic_colors = np.array([colors[label] for label in data["tracked_points"].labels]) * 255

In [ ]:
if "server" not in globals():
    server = viser.ViserServer(port=8080)

background_img = np.zeros((1080, 1920, 3), dtype=np.uint8)  # RGB black
server.scene.set_background_image(background_img, format="jpeg")
hanlde = server.scene.add_point_cloud(
    "/semantic_cloud",
    points=points.astype(np.float32),
    # colors=depth_colors.astype(np.uint8),
    colors=semantic_colors.astype(np.uint8),
    # colors=colors,
    point_size=0.02,  # adjust as needed
    point_shape="circle",
)

## Path Planning


In [ ]:
import plotly.graph_objects as go
from lac.planning.arc_planner import ArcPlanner

from lac.utils.plotting import (
    plot_points_rover_frame,
    plot_path_rover_frame,
    plot_rocks_rover_frame,
)

In [ ]:
planner = ArcPlanner(arc_duration=3.0)

fig = go.Figure()
for arc in planner.candidate_arcs:
    fig = plot_path_rover_frame(arc, fig=fig, color="teal", showlegend=False)

fig.update_layout(
    height=800,
    width=1200,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=0, r=0, t=0, b=0),
)
fig.show()

In [ ]:
data["rock_data"]

## Feature extraction


In [ ]:
from lightglue import viz2d
from lightglue.utils import rbd

In [ ]:
frame = END_FRAME
image = images["FrontRight"][frame]
feats = feature_tracker.extract_feats(image)
feats = rbd(feats)
viz2d.plot_images([image])
viz2d.plot_keypoints([feats["keypoints"]], colors=["lime"], ps=10)

## Feature matching


In [ ]:
frame = 82
left_image = images["FrontLeft"][frame]
right_image = images["FrontRight"][frame]

In [ ]:
start_time = time.time()
feats1 = feature_tracker.extract_feats(left_image)
print(f"feature extraction time: {time.time() - start_time} seconds")
feats2 = feature_tracker.extract_feats(right_image)
start_time = time.time()
matches = feature_tracker.match_feats(feats1, feats2)
end_time = time.time()
print(f"matching time: {end_time - start_time} seconds")

points1 = feats1["keypoints"][0][matches[:, 0]].cpu().numpy()
points2 = feats2["keypoints"][0][matches[:, 1]].cpu().numpy()

viz2d.plot_images([left_image, right_image], pad=0.0)
viz2d.plot_matches(points1, points2, color="lime", lw=0.2)

In [ ]:
# Convert numpy arrays to PIL images
# Convert grayscale numpy arrays to RGB PIL images
left_image_rgb = np.stack([left_image, left_image, left_image], axis=-1)
right_image_rgb = np.stack([right_image, right_image, right_image], axis=-1)
left_image_pil = Image.fromarray(left_image_rgb)
right_image_pil = Image.fromarray(right_image_rgb)

In [ ]:
from transformers import AutoImageProcessor, AutoModel
import torch

processor = AutoImageProcessor.from_pretrained("ETH-CVG/lightglue_superpoint")
model = AutoModel.from_pretrained("ETH-CVG/lightglue_superpoint")

In [ ]:
start_time = time.time()
images = [left_image_pil, right_image_pil]
inputs = processor(images, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

image_sizes = [[(im.height, im.width) for im in images]]
matches_hf = processor.post_process_keypoint_matching(outputs, image_sizes, threshold=0.2)
print(f"Time taken: {time.time() - start_time} seconds")

In [ ]:
len(matches)

In [ ]:
len(matches_hf[0]["matching_scores"])

# Segmentation

In [ ]:
semantic_images = load_images(
    data_path,
    cameras=["FrontLeft_semantic", "FrontRight_semantic"],
    start_frame=0,
    end_frame=10000,
)

# Compute


In [ ]:
from pypapi import papi_low as papi
from pypapi import events

papi.library_init()

evs = papi.create_eventset()
papi.add_event(evs, events.PAPI_FP_OPS)

papi.start(evs)

# Do some computation here

result = papi.stop(evs)
print(result)

papi.cleanup_eventset(evs)
papi.destroy_eventset(evs)

In [ ]:
from pypapi import events as ev, papi_high as high


def ok(e):
    try:
        high.start_counters([e])
        high.stop_counters()
        return True
    except Exception:
        return False


for e in [ev.PAPI_DP_OPS, ev.PAPI_SP_OPS, ev.PAPI_FP_OPS, ev.PAPI_TOT_INS, ev.PAPI_TOT_CYC]:
    print(e, ok(e))

In [ ]:
from pypapi import papi_high

papi_high.hl_region_begin("computation")

# computation

papi_high.hl_region_end("computation")

In [ ]:
from pypapi import events as papi_events
from pypapi import papi_high as high
import time

# Choose the event(s) you want:
EVENTS = [papi_events.PAPI_DP_OPS]  # or PAPI_SP_OPS / PAPI_FP_OPS


def do_work(n=10_000_00):
    # something with FP ops
    x = 0.0
    for i in range(n):
        x = x * 1.0000001 + 1.0
    return x


def count_flops():
    # Warm-up to stabilize JITed libs, cache, etc.
    do_work(1000)

    # Measure wall time alongside counters (for FLOP/s)
    t0 = time.perf_counter()
    high.start(EVENTS)
    do_work()
    counts = high.stop_counters()
    t1 = time.perf_counter()

    flops = counts[0]
    secs = t1 - t0
    print(f"FLOPs: {flops:,}")
    print(f"Time:  {secs:.6f} s")
    if secs > 0:
        print(f"FLOP/s: {flops / secs:,.0f}")


if __name__ == "__main__":
    try:
        count_flops()
    except Exception as e:
        print("Profiling failed. Common causes:")
        print("- The chosen PAPI event isn’t supported on your CPU")
        print("- PAPI not installed with dev headers")
        print("Error:", e)
